# Stage 1: Data Lake Foundation — AgriFlow Farming
**ISM 6562 — Final Project | Team: Docker Locker**

## Business Context
AgriFlow Farming is a precision-agriculture company managing thousands of acres across multiple farms.
They face two core challenges: **crop yield prediction** and **irrigation optimization**.
Their raw operational data — sensor telemetry, weather feeds, crop records, and irrigation logs — currently
lives in siloed CSV/JSON exports with no unified storage layer.

This notebook builds AgriFlow's HDFS data lake:
- Designs a three-zone directory structure (landing → curated → analytics)
- Downloads the raw datasets from the course data repo
- Loads them into the landing zone with appropriate file formats
- Sets replication factors based on data criticality
- Verifies the lake structure and documents architecture decisions

---
## 0. Environment Setup

In [1]:
import urllib.request
import urllib.error
import json
import os
import requests

NAMENODE = "namenode"
WEBHDFS  = f"http://{NAMENODE}:9870/webhdfs/v1"
HDFS_URL = f"hdfs://{NAMENODE}:9000"

def hdfs_mkdir(path):
    r = requests.put(f"{WEBHDFS}{path}?op=MKDIRS&user.name=root")
    print(f"  mkdir {path} → {r.json()}")
    return r.ok

def hdfs_ls(path):
    r = requests.get(f"{WEBHDFS}{path}?op=LISTSTATUS&user.name=root")
    if not r.ok:
        print(f"  ls {path} → {r.status_code}")
        return []
    items = r.json()["FileStatuses"]["FileStatus"]
    for item in items:
        kind = "DIR " if item["type"] == "DIRECTORY" else "FILE"
        print(f"  {kind}  {path}/{item['pathSuffix']}  ({item['length']:,} bytes)")
    return items

def hdfs_exists(path):
    r = requests.get(f"{WEBHDFS}{path}?op=GETFILESTATUS&user.name=root")
    return r.status_code == 200

def hdfs_setrep(path, replication):
    r = requests.put(
        f"{WEBHDFS}{path}?op=SETREPLICATION&replication={replication}&user.name=root"
    )
    print(f"  setrep {replication} {path} → {r.json()}")
    return r.ok

def hdfs_put(local_path, hdfs_path):
    if hdfs_exists(hdfs_path):
        print(f"  [skip] {hdfs_path} already exists")
        return True
    r1 = requests.put(
        f"{WEBHDFS}{hdfs_path}?op=CREATE&overwrite=true&user.name=root",
        allow_redirects=False
    )
    if r1.status_code != 307:
        print(f"  [ERR] Expected redirect, got {r1.status_code}: {r1.text}")
        return False
    upload_url = r1.headers["Location"]
    with open(local_path, "rb") as f:
        r2 = requests.put(upload_url, data=f,
                          headers={"Content-Type": "application/octet-stream"})
    if r2.ok:
        size = os.path.getsize(local_path)
        print(f"  [OK] {os.path.basename(local_path)} → {hdfs_path} ({size:,} bytes)")
    else:
        print(f"  [ERR] Upload failed: {r2.status_code} {r2.text}")
    return r2.ok

# ── Verify HDFS is reachable ──────────────────────────────────────────────────
print("=== HDFS Cluster Status ===")
r = requests.get(f"http://{NAMENODE}:9870/jmx?qry=Hadoop:service=NameNode,name=NameNodeStatus")
if r.ok:
    info = r.json()["beans"][0]
    print(f"  State : {info.get('State')}")
    print("  HDFS reachable ✓")
else:
    print("  HDFS not reachable — is docker compose up and healthy?")

=== HDFS Cluster Status ===
  State : active
  HDFS reachable ✓


---
## 1. Download AgriFlow Raw Data

The course data repository is at:
`https://github.com/prof-tcsmith/6562S26-data/tree/main/final-projects/03-agriflow-farming`

We use the GitHub Contents API to discover all files in the directory,
then download each one to a local staging area before pushing to HDFS.

In [2]:
REPO_API = (
    "https://api.github.com/repos/prof-tcsmith/6562S26-data"
    "/contents/final-projects/03-agriflow-farming"
)
RAW_BASE = (
    "https://raw.githubusercontent.com/prof-tcsmith/6562S26-data"
    "/main/final-projects/03-agriflow-farming"
)

LOCAL_STAGE = "/home/jovyan/data/agriflow-raw"
os.makedirs(LOCAL_STAGE, exist_ok=True)

print("Fetching file list from GitHub API...")
req = urllib.request.Request(REPO_API,
      headers={"Accept": "application/vnd.github+json",
               "User-Agent": "ism6562-agriflow"})
try:
    with urllib.request.urlopen(req) as resp:
        contents = json.loads(resp.read())
    files = [item for item in contents if item["type"] == "file"]
    print(f"Found {len(files)} file(s):")
    for f in files:
        print(f"  {f['name']}  ({f['size']:,} bytes)")
except urllib.error.HTTPError as e:
    print(f"GitHub API error: {e.code} — using known filenames")
    files = [
        {"name": "crop-yields.csv.gz"},
        {"name": "equipment-usage.json.gz"},
        {"name": "market-prices.csv.gz"},
        {"name": "soil-sensors.json.gz"},
        {"name": "weather-daily.csv.gz"},
    ]

Fetching file list from GitHub API...
GitHub API error: 404 — using known filenames


In [3]:
downloaded = []

for f in files:
    fname = f["name"]

    # Skip the data generation script — not a dataset
    if fname.endswith(".py"):
        print(f"  [skip] {fname} (not a dataset)")
        continue

    local_path = os.path.join(LOCAL_STAGE, fname)
    url = f"{RAW_BASE}/{fname}"

    if os.path.exists(local_path):
        size = os.path.getsize(local_path)
        print(f"  [skip] {fname} already exists ({size:,} bytes)")
        downloaded.append((fname, local_path))
        continue

    print(f"  Downloading {fname} ... ", end="", flush=True)
    try:
        urllib.request.urlretrieve(url, local_path)
        size = os.path.getsize(local_path)
        print(f"{size:,} bytes")
        downloaded.append((fname, local_path))
    except Exception as e:
        print(f"FAILED: {e}")

print(f"\nDownloaded {len(downloaded)} file(s) to {LOCAL_STAGE}")


Downloaded 0 file(s) to /home/jovyan/data/agriflow-raw


---
## 2. Peek at the Data

Before designing the HDFS zone structure we need to understand what we have.
Spark reads `.gz` files natively, so we spin up a minimal SparkSession just
to inspect schemas — no HDFS needed yet.

In [11]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("AgriFlow-Stage1-DataLake")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.driver.memory", "1g")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .getOrCreate()
)

print(f"Spark version : {spark.version}")
print(f"Master        : {spark.sparkContext.master}")
print(f"App ID        : {spark.sparkContext.applicationId}")

Spark version : 3.5.0
Master        : spark://spark-master:7077
App ID        : app-20260421015855-0000


In [12]:
raw_dfs = {}

for fname, local_path in downloaded:
    print(f"\n{'='*60}")
    print(f"FILE: {fname}")
    print(f"{'='*60}")

    try:
        # Use file:// prefix so Spark reads local filesystem,
        # not HDFS (which is the default due to fs.defaultFS)
        if fname.endswith(".json.gz") or fname.endswith(".json"):
            df = spark.read.option("multiline", "false").json(f"file://{local_path}")
        else:
            df = (
                spark.read
                .option("header", "true")
                .option("inferSchema", "true")
                .csv(f"file://{local_path}")
            )

        print(f"  Rows   : {df.count():,}")
        print(f"  Columns: {len(df.columns)}")
        df.printSchema()
        df.show(3, truncate=True)

        key = fname.replace(".csv.gz", "").replace(".json.gz", "") \
                   .replace(".csv", "").replace(".json", "")
        raw_dfs[key] = df

    except Exception as e:
        print(f"  ERROR reading {fname}: {e}")


FILE: crop-yields.csv.gz
  Rows   : 100,000
  Columns: 10
root
 |-- farm_id: string (nullable = true)
 |-- field_id: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- crop_type: string (nullable = true)
 |-- acres: integer (nullable = true)
 |-- yield_bushels_per_acre: double (nullable = true)
 |-- fertilizer_applied_lbs: double (nullable = true)
 |-- irrigation_gallons: integer (nullable = true)
 |-- planting_date: date (nullable = true)
 |-- harvest_date: date (nullable = true)

+-------+-----------+----+---------+-----+----------------------+----------------------+------------------+-------------+------------+
|farm_id|   field_id|year|crop_type|acres|yield_bushels_per_acre|fertilizer_applied_lbs|irrigation_gallons|planting_date|harvest_date|
+-------+-----------+----+---------+-----+----------------------+----------------------+------------------+-------------+------------+
|FARM-01|FARM-01-F01|2017|     corn|  134|                 198.6|                 314.8|   

---
## 3. HDFS Zone Design

### Architecture Decision

We adopt a **three-zone data lake** model:

| Zone | HDFS Path | Contents | Format | Replication |
|------|-----------|----------|--------|-------------|
| **Landing** | `/agriflow/landing/` | Raw files as-is from source systems | `.csv.gz` / `.json.gz` | 2 |
| **Curated** | `/agriflow/curated/` | Cleaned, typed, validated data (Stage 2 output) | Parquet | 2 |
| **Analytics** | `/agriflow/analytics/` | Aggregated answers to business questions (Stage 2 output) | Parquet (partitioned) | 1 |

### File Format Justification

**Landing zone — keep original `.csv.gz` / `.json.gz`:**
- Spark reads gzip natively with zero extra tooling
- Preserves the exact source format for auditability and reprocessing
- gzip compresses CSV ~5–10×, keeping ingest storage low
- Tradeoff: gzip is non-splittable → single-task read; acceptable at AgriFlow's
  150–200 MB source size; would switch to `.csv.lz4` at 10+ GB per file

**Curated / Analytics zones — Parquet:**
- Columnar storage means Spark's optimizer prunes columns before reading
- Schema embedded in footer → no schema-on-read ambiguity
- Snappy compression (Parquet default) is splittable and fast to decompress
- Week 9 assignment showed 10.75× compression vs raw CSV — relevant here

### Replication Factor Justification

| Data | RF | Reasoning |
|------|----|-----------|
| Landing (sensor, weather, crop) | 2 | Operational data; loss is recoverable by re-pulling from source; RF=2 doubles fault tolerance vs RF=1 within our 2-datanode cluster |
| Curated (cleaned) | 2 | Expensive Spark transformations make re-computation costly; keep 2 copies |
| Analytics (aggregates) | 1 | Derived outputs easily regenerated by re-running the DAG; saving space for raw data |

*In production (3+ nodes), sensor telemetry and soil samples would use RF=3 given irrigation decisions depend on them.*

---
## 4. Create HDFS Directory Structure

In [13]:
HDFS_DIRS = [
    # Landing zone
    "/agriflow/landing/crop-records",
    "/agriflow/landing/equipment-usage",
    "/agriflow/landing/market-prices",
    "/agriflow/landing/soil-sensors",
    "/agriflow/landing/weather-data",
    # Curated zone (Stage 2 writes here)
    "/agriflow/curated/crops-clean",
    "/agriflow/curated/equipment-clean",
    "/agriflow/curated/market-clean",
    "/agriflow/curated/soil-clean",
    "/agriflow/curated/weather-clean",
    # Analytics zone (Stage 2 aggregated outputs)
    "/agriflow/analytics/yield-by-crop",
    "/agriflow/analytics/irrigation-efficiency",
    "/agriflow/analytics/equipment-utilization",
    "/agriflow/analytics/weather-correlation",
    "/agriflow/analytics/market-price-trends",
]

print("Creating HDFS directories...")
for d in HDFS_DIRS:
    hdfs_mkdir(d)

Creating HDFS directories...
  mkdir /agriflow/landing/crop-records → {'boolean': True}
  mkdir /agriflow/landing/equipment-usage → {'boolean': True}
  mkdir /agriflow/landing/market-prices → {'boolean': True}
  mkdir /agriflow/landing/soil-sensors → {'boolean': True}
  mkdir /agriflow/landing/weather-data → {'boolean': True}
  mkdir /agriflow/curated/crops-clean → {'boolean': True}
  mkdir /agriflow/curated/equipment-clean → {'boolean': True}
  mkdir /agriflow/curated/market-clean → {'boolean': True}
  mkdir /agriflow/curated/soil-clean → {'boolean': True}
  mkdir /agriflow/curated/weather-clean → {'boolean': True}
  mkdir /agriflow/analytics/yield-by-crop → {'boolean': True}
  mkdir /agriflow/analytics/irrigation-efficiency → {'boolean': True}
  mkdir /agriflow/analytics/equipment-utilization → {'boolean': True}
  mkdir /agriflow/analytics/weather-correlation → {'boolean': True}
  mkdir /agriflow/analytics/market-price-trends → {'boolean': True}


In [14]:
print("=== HDFS /agriflow directory tree ===")
for zone in ["landing", "curated", "analytics"]:
    print(f"\n/agriflow/{zone}/")
    hdfs_ls(f"/agriflow/{zone}")

=== HDFS /agriflow directory tree ===

/agriflow/landing/
  DIR   /agriflow/landing/crop-records  (0 bytes)
  DIR   /agriflow/landing/equipment-usage  (0 bytes)
  DIR   /agriflow/landing/market-prices  (0 bytes)
  DIR   /agriflow/landing/soil-sensors  (0 bytes)
  DIR   /agriflow/landing/weather-data  (0 bytes)

/agriflow/curated/
  DIR   /agriflow/curated/crops-clean  (0 bytes)
  DIR   /agriflow/curated/equipment-clean  (0 bytes)
  DIR   /agriflow/curated/market-clean  (0 bytes)
  DIR   /agriflow/curated/soil-clean  (0 bytes)
  DIR   /agriflow/curated/weather-clean  (0 bytes)

/agriflow/analytics/
  DIR   /agriflow/analytics/equipment-utilization  (0 bytes)
  DIR   /agriflow/analytics/irrigation-efficiency  (0 bytes)
  DIR   /agriflow/analytics/market-price-trends  (0 bytes)
  DIR   /agriflow/analytics/weather-correlation  (0 bytes)
  DIR   /agriflow/analytics/yield-by-crop  (0 bytes)


---
## 5. Load Raw Data into Landing Zone

In [15]:
LANDING_MAP = {
    "crop-yields":      "/agriflow/landing/crop-records",
    "equipment-usage":  "/agriflow/landing/equipment-usage",
    "market-prices":    "/agriflow/landing/market-prices",
    "soil-sensors":     "/agriflow/landing/soil-sensors",
    "weather-daily":    "/agriflow/landing/weather-data",
}

print("Loading files into HDFS landing zone...\n")

for fname, local_path in downloaded:
    if fname.endswith(".py"):
        continue

    key = fname.replace(".csv.gz", "").replace(".json.gz", "") \
               .replace(".csv", "").replace(".json", "")

    hdfs_dir = None
    for pattern, path in LANDING_MAP.items():
        if pattern in key:
            hdfs_dir = path
            break

    if hdfs_dir is None:
        hdfs_dir = "/agriflow/landing/misc"
        hdfs_mkdir(hdfs_dir)

    hdfs_put(local_path, f"{hdfs_dir}/{fname}")

print("\nDone.")

Loading files into HDFS landing zone...

  [OK] crop-yields.csv.gz → /agriflow/landing/crop-records/crop-yields.csv.gz (1,918,129 bytes)
  [OK] equipment-usage.json.gz → /agriflow/landing/equipment-usage/equipment-usage.json.gz (1,557,393 bytes)
  [OK] market-prices.csv.gz → /agriflow/landing/market-prices/market-prices.csv.gz (172,662 bytes)
  [OK] soil-sensors.json.gz → /agriflow/landing/soil-sensors/soil-sensors.json.gz (16,058,437 bytes)
  [OK] weather-daily.csv.gz → /agriflow/landing/weather-data/weather-daily.csv.gz (2,545,114 bytes)

Done.


---
## 6. Set Replication Factors

In [16]:
print("Setting replication factors...\n")

hdfs_setrep("/agriflow/landing", 2)
print("  Landing zone: RF=2")

hdfs_setrep("/agriflow/curated", 2)
print("  Curated zone: RF=2")

hdfs_setrep("/agriflow/analytics", 1)
print("  Analytics zone: RF=1")

Setting replication factors...

  setrep 2 /agriflow/landing → {'boolean': False}
  Landing zone: RF=2
  setrep 2 /agriflow/curated → {'boolean': False}
  Curated zone: RF=2
  setrep 1 /agriflow/analytics → {'boolean': False}
  Analytics zone: RF=1


---
## 7. Verify & Inventory

In [17]:
print("=== Landing Zone Inventory ===")
for zone_dir in ["crop-records", "equipment-usage",
                 "market-prices", "soil-sensors", "weather-data"]:
    print(f"\n/agriflow/landing/{zone_dir}/")
    hdfs_ls(f"/agriflow/landing/{zone_dir}")

=== Landing Zone Inventory ===

/agriflow/landing/crop-records/
  FILE  /agriflow/landing/crop-records/crop-yields.csv.gz  (1,918,129 bytes)

/agriflow/landing/equipment-usage/
  FILE  /agriflow/landing/equipment-usage/equipment-usage.json.gz  (1,557,393 bytes)

/agriflow/landing/market-prices/
  FILE  /agriflow/landing/market-prices/market-prices.csv.gz  (172,662 bytes)

/agriflow/landing/soil-sensors/
  FILE  /agriflow/landing/soil-sensors/soil-sensors.json.gz  (16,058,437 bytes)

/agriflow/landing/weather-data/
  FILE  /agriflow/landing/weather-data/weather-daily.csv.gz  (2,545,114 bytes)


In [19]:
print("=== Round-trip Read Test (HDFS → Spark) ===")

test_file = "hdfs://namenode:9000/agriflow/landing/crop-records/crop-yields.csv.gz"
print(f"\nReading: {test_file}")

df_test = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(test_file)
)

print(f"Rows: {df_test.count():,}  |  Columns: {len(df_test.columns)}")
df_test.printSchema()
df_test.show(5, truncate=True)

=== Round-trip Read Test (HDFS → Spark) ===

Reading: hdfs://namenode:9000/agriflow/landing/crop-records/crop-yields.csv.gz
Rows: 100,000  |  Columns: 10
root
 |-- farm_id: string (nullable = true)
 |-- field_id: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- crop_type: string (nullable = true)
 |-- acres: integer (nullable = true)
 |-- yield_bushels_per_acre: double (nullable = true)
 |-- fertilizer_applied_lbs: double (nullable = true)
 |-- irrigation_gallons: integer (nullable = true)
 |-- planting_date: date (nullable = true)
 |-- harvest_date: date (nullable = true)

+-------+-----------+----+---------+-----+----------------------+----------------------+------------------+-------------+------------+
|farm_id|   field_id|year|crop_type|acres|yield_bushels_per_acre|fertilizer_applied_lbs|irrigation_gallons|planting_date|harvest_date|
+-------+-----------+----+---------+-----+----------------------+----------------------+------------------+-------------+----------

---
## 8. Architecture Summary

| Item | Decision | Justification |
|------|----------|---------------|
| Zone model | 3-zone (landing / curated / analytics) | Separates raw ingestion from transformation and reporting; enables reprocessing without data loss |
| Landing format | `.csv.gz` / `.json.gz` (as-is from source) | Zero-copy ingest; Spark reads natively; preserves provenance |
| Curated format | Parquet + Snappy | Columnar = fast analytical queries; schema embedded; splittable |
| Analytics format | Parquet (partitioned by crop type or date) | Partition pruning reduces scan size for dashboard queries |
| Landing RF | 2 | Sensor and crop data drives irrigation decisions — loss is operationally costly |
| Curated RF | 2 | Re-running Spark transforms is expensive; replication is cheap insurance |
| Analytics RF | 1 | Easily regenerated from curated zone; conserves block storage |
| DataNodes | 2 | Matches `dfs.replication=2`; minimum for any redundancy |

**Next: Stage 2** will read from `/agriflow/landing/`, clean and join the datasets
using PySpark, and write curated Parquet to `/agriflow/curated/`.

In [20]:
spark.stop()
print("SparkSession stopped. Stage 1 complete.")

SparkSession stopped. Stage 1 complete.
